# Semantic chunking with Jev: cut where the topic changes

Most semantic chunkers cut where the embeddings of neighbouring sentences drift apart. That measures word overlap, not whether the text moved on: a paragraph that changes vocabulary mid-argument gets split, and two articles that share vocabulary get merged.

`recipes/chunker` asks Jev, TypeSafe's decision model, a different question. Jev reads a window of numbered sentences and judges each gap: **does the next sentence start a new topic?** The answers are probabilities, so a single set of Jev requests serves every threshold, and cutting is a pure function of those probabilities plus a minimum and maximum chunk size.

This notebook runs it on Chroma's chunking benchmark and answers one question:

**At the same mean chunk size, do Jev's chunks retrieve the answer passages better than fixed-size, recursive and embedding-similarity chunks?**

Chunk size alone moves every retrieval score, so Jev's threshold is tuned to match each baseline's mean chunk size, and each comparison is made only at that matched size.

## Which gap judge runs

- **With `OPENROUTER_API_KEY` set** (in the environment or in the repo's `.env`), Jev (`typesafe/jev-1.13`) judges every gap through OpenRouter's System One endpoint. Each request is one window of 40 numbered sentences, and the window advances 20 sentences at a time. Each gap is judged in the window where it sits nearest the centre, so Jev always sees context on both sides of it. A pilot on WANDS search results showed Jev answers per-item questions over a numbered list reliably: each item kept its answer when the list was reversed (correlation 0.98), and planted duplicates were caught at 0.95 to 0.99. A request here is about 2,900 tokens, roughly $0.00013, and the whole benchmark is 635 requests, about $0.08 to $0.11. The notebook judges the smallest corpus first, extrapolates the cost, and stops if the estimate exceeds `JEVQU_MAX_USD` (default $1). Answers are cached in `.jev_cache/`, so reruns cost nothing.
- **Without a key**, a **stand-in** answers the same questions: P(new topic) = 1 − cosine similarity of the two sentences' bge-small embeddings. **The stand-in is not Jev.** In that mode the results measure the pipeline and an embedding-dissimilarity chunker, and say nothing about Jev.

In [1]:
import os, sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore", message="Cannot enable progress bars")

import numpy as np
import tiktoken
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "jevqu").is_dir())
sys.path.insert(0, str(ROOT))
for line in (ROOT / ".env").read_text().splitlines() if (ROOT / ".env").exists() else []:
    key, _, value = line.partition("=")
    if key.strip() and not key.lstrip().startswith("#"):
        os.environ.setdefault(key.strip(), value.strip().strip("'\""))

from jevqu import JEV_MODEL
from jevqu.jev import Jev
from jevqu.metrics import paired_bootstrap
from recipes.chunker.chunker import MAX_SENTENCE_CHARS, chunks, cut, gap_probs, sentences, windows

WINDOW = 40        # sentences per Jev request; the window advances WINDOW // 2 sentences at a time
RETRIEVE = 5       # chunks retrieved per question
MAX_TOKENS = 512   # bge-small embeds at most 512 tokens, so Jev's chunks are capped there
WORKERS = 8
MAX_USD = float(os.environ.get("JEVQU_MAX_USD", "1"))
USE_JEV = bool(os.environ.get("OPENROUTER_API_KEY"))
JUDGE = f"Jev ({JEV_MODEL})" if USE_JEV else "stand-in (embedding dissimilarity, NOT Jev)"
print("gap judge:", JUDGE)

gap judge: Jev (typesafe/jev-1.13)


## 1. The benchmark

[Chroma's chunking evaluation](https://github.com/brandonstarxel/chunking_evaluation) has five corpora (chat logs, financial filings, PubMed abstracts, a State of the Union address, Wikipedia articles) and 472 questions, each with the reference passages that answer it. A chunker is scored by embedding its chunks, retrieving 5 per question, and measuring token overlap with the references:

- **recall**: share of reference tokens that appear in the retrieved chunks;
- **precision**: share of retrieved tokens that belong to a reference;
- **IoU**: overlap divided by the union of retrieved and reference tokens. It rewards chunks that hold the answer and little else.

Two adjustments to the package:

- **Local embeddings.** The package defaults to OpenAI embeddings. Here everything is embedded locally with `BAAI/bge-small-en-v1.5` through FastEmbed, so absolute scores differ from Chroma's report.
- **A chromadb fix.** chromadb 1.x raises `NotFoundError` where the package expects `ValueError`, so its first run crashes. Pre-creating the package's two scratch collections avoids that.

In [2]:
from chromadb import EmbeddingFunction
from fastembed import TextEmbedding
from chunking_evaluation import GeneralEvaluation
from chunking_evaluation.chunking import FixedTokenChunker, RecursiveTokenChunker
from chonkie import SemanticChunker

enc = tiktoken.get_encoding("cl100k_base")
def ntok(s):
    return len(enc.encode(s, disallowed_special=()))

class FastEmbedFunction(EmbeddingFunction):
    # Local embeddings for chromadb, memoized: the 472 questions are embedded once, not once per run.
    def __init__(self, model="BAAI/bge-small-en-v1.5"):
        self.model, self.memo = TextEmbedding(model), {}
    def __call__(self, input):
        todo = [t for t in dict.fromkeys(input) if t not in self.memo]
        self.memo.update(zip(todo, self.model.embed(todo, batch_size=64)))
        return [self.memo[t] for t in input]

ef = FastEmbedFunction()
ev = GeneralEvaluation()
for name in ("auto_chunk", "auto_questions"):  # chromadb >= 1.0 raises NotFoundError where the package expects ValueError
    ev.chroma_client.get_or_create_collection(name)
corpora = {cid: Path(p).read_text() for cid, p in sorted(ev.corpora_id_paths.items())}
nq = ev.questions_df["corpus_id"].value_counts()
rows = ["| corpus | characters | tokens | questions |", "|---|---|---|---|"]
rows += [f"| {cid} | {len(t):,} | {ntok(t):,} | {nq[cid]} |" for cid, t in corpora.items()]
display(Markdown("\n".join(rows)))

| corpus | characters | tokens | questions |
|---|---|---|---|
| chatlogs | 40,000 | 7,727 | 56 |
| finance | 737,905 | 166,177 | 97 |
| pubmed | 500,000 | 117,211 | 99 |
| state_of_the_union | 48,051 | 10,444 | 76 |
| wikitexts | 118,372 | 26,649 | 144 |

## 2. Sentences and gap probabilities

Sentences are split with a regex on terminal punctuation and newlines; every chunk is an exact slice of the corpus, as the benchmark requires. Each gap gets one probability, P(the next sentence starts a new topic).

In [3]:
docs = {}
for cid, text in corpora.items():
    spans = sentences(text)
    texts = [text[a:b] for a, b in spans]
    docs[cid] = {"spans": spans, "texts": texts, "sizes": [ntok(t) for t in texts]}
n_req = sum(len(windows(len(d["texts"]), WINDOW)) for d in docs.values())
print(f"{sum(len(d['texts']) for d in docs.values()):,} sentences, {n_req} windows of {WINDOW}")

class StandIn:
    # Offline stand-in with Jev's ask() contract. NOT Jev: P(new topic) = 1 - cosine of the two sentences' embeddings.
    model = "stand-in"
    def __init__(self):
        uniq = list(dict.fromkeys(t[:MAX_SENTENCE_CHARS] for d in docs.values() for t in d["texts"]))
        vecs = np.array(list(ef.model.embed(uniq, batch_size=64)))
        self.vec = dict(zip(uniq, vecs / np.linalg.norm(vecs, axis=1, keepdims=True)))
    def ask(self, state, questions):
        t = {s["sentence"]: s["text"] for s in state["sentences"]}
        return {q: {"type": "noul", "noul": float(1 - self.vec[t[k]] @ self.vec[t[k + 1]])}
                for q, k in ((q, int(q.split("_")[1])) for q in questions)}

t0 = time.time()
judge = Jev(cache_dir=ROOT / ".jev_cache") if USE_JEV else StandIn()
probs = {}
if USE_JEV:  # price the smallest corpus first and stop before the full run if it would cost too much
    pilot = min(docs, key=lambda c: len(corpora[c]))
    probs[pilot] = gap_probs(docs[pilot]["texts"], judge, WINDOW, WORKERS)
    # ponytail: estimate from uncached requests only; a pilot already in the cache estimates $0
    est = float(np.mean(judge.usage)) * n_req if judge.usage else 0.0
    print(f"pilot on {pilot}: ${sum(judge.usage):.4f} for {len(judge.usage)} requests; about ${est:.2f} for all {n_req}")
    assert est <= MAX_USD, f"estimate is above JEVQU_MAX_USD=${MAX_USD:.2f}"
for cid, d in docs.items():
    if cid not in probs:
        probs[cid] = gap_probs(d["texts"], judge, WINDOW, WORKERS)
print(f"judged {sum(map(len, probs.values())):,} gaps in {time.time() - t0:.0f}s with {JUDGE}")
if USE_JEV:
    print(f"spent ${sum(judge.usage):.4f} on {len(judge.usage)} uncached Jev requests")

12,783 sentences, 635 windows of 40
pilot on chatlogs: $0.0000 for 0 requests; about $0.00 for all 635
judged 12,778 gaps in 0s with Jev (typesafe/jev-1.13)
spent $0.0000 on 0 uncached Jev requests


### What the judge sees

A stretch of the Wikipedia corpus around its most confident gap. Each line is a sentence, preceded by P(this sentence starts a new topic).

In [4]:
p, texts = probs["wikitexts"], docs["wikitexts"]["texts"]
g = int(np.argmax(p[:400]))
for i in range(max(1, g - 8), min(len(texts), g + 9)):
    mark = "<<" if i == g + 1 else "  "
    print(f"{p[i - 1]:.2f} {mark} {texts[i][:100]}")

0.14    During gameplay , characters will call out if something happens to them , such as their health point
0.30    Each character has specific " Potentials " , skills unique to each character .
0.05    They are divided into " Personal Potential " , which are innate skills that remain unaltered unless 
0.10    To learn Battle Potentials , each character has a unique " Masters Table " , a grid @-@ based skill 
0.28    Characters also have Special Abilities that grant them temporary boosts on the battlefield : Kurt ca
0.32    Troops are divided into five classes : Scouts , <unk> , Engineers , Lancers and Armored Soldier .
0.06    Troopers can switch classes by changing their assigned weapon .
0.05    Changing class does not greatly affect the stats gained while in a previous class .
0.21    With victory in battle , experience points are awarded to the squad , which are distributed into fiv
0.98 << = = Plot = =
0.35    The game takes place during the Second Europan War .
0.39    Gallian 

## 3. Baselines

- **fixed**: 200 and 400 tokens, no overlap (the package's `FixedTokenChunker`);
- **recursive**: up to 200 and 400 tokens, splitting on paragraphs, lines, then sentence punctuation (`RecursiveTokenChunker`, with token length rather than its default character length);
- **chonkie semantic**: chonkie's `SemanticChunker` with its default embedding model (`minishlab/potion-base-32M`), at its default similarity threshold 0.8 and at 0.5, with chunks capped at 512 tokens.

Sizes are cl100k tokens.

In [5]:
def recursive(n):
    return RecursiveTokenChunker(chunk_size=n, chunk_overlap=0, length_function=ntok)

class Chonkie:
    def __init__(self, threshold):
        self.chunker = SemanticChunker(threshold=threshold, chunk_size=MAX_TOKENS)
    def split_text(self, text):
        return [c.text for c in self.chunker.chunk(text)]

baselines = {"fixed 200": FixedTokenChunker(chunk_size=200, chunk_overlap=0),
             "fixed 400": FixedTokenChunker(chunk_size=400, chunk_overlap=0),
             "recursive 200": recursive(200), "recursive 400": recursive(400),
             "chonkie semantic, threshold 0.8": Chonkie(0.8), "chonkie semantic, threshold 0.5": Chonkie(0.5)}
base_sizes = {name: [ntok(c) for t in corpora.values() for c in ch.split_text(t)] for name, ch in baselines.items()}
for name, s in base_sizes.items():
    print(f"{name:32} {len(s):5} chunks, mean {np.mean(s):5.0f} tokens, max {max(s)}")

fixed 200                         1644 chunks, mean   200 tokens, max 200
fixed 400                          824 chunks, mean   398 tokens, max 400
recursive 200                     2386 chunks, mean   137 tokens, max 200
recursive 400                     1187 chunks, mean   276 tokens, max 400
chonkie semantic, threshold 0.8   3722 chunks, mean    89 tokens, max 663
chonkie semantic, threshold 0.5   2401 chunks, mean   137 tokens, max 684


## 4. Jev chunks at matched sizes

For each baseline, the threshold is bisected until the mean size of Jev's chunks matches the baseline's mean. The minimum chunk size is a quarter of the target and the maximum is 512 tokens. Cuts forced by the 512-token cap are counted separately: when most cuts are forced, the judge is barely deciding anything.

In [6]:
by_text = {t: cid for cid, t in corpora.items()}

class JevChunks:
    # split_text() for the benchmark: cuts the precomputed gap probabilities, no new judge calls.
    def __init__(self, threshold, min_size):
        self.threshold, self.min_size = threshold, min_size
    def starts(self, cid):
        return cut(probs[cid], docs[cid]["sizes"], self.threshold, self.min_size, MAX_TOKENS)
    def split_text(self, text):
        cid = by_text[text]
        return chunks(text, docs[cid]["spans"], self.starts(cid))
    def mean_size(self):  # sum of sentence tokens per chunk: fast, within a few tokens of ntok(chunk)
        sizes = []
        for c in docs:
            s = self.starts(c)
            sizes += [sum(docs[c]["sizes"][a:b]) for a, b in zip(s, s[1:] + [len(docs[c]["sizes"])])]
        return float(np.mean(sizes))
    def judged_share(self):  # share of cuts the judge made, as opposed to cuts forced by MAX_TOKENS
        cuts = [probs[c][i - 1] >= self.threshold for c in docs for i in self.starts(c)[1:]]
        return float(np.mean(cuts)) if cuts else 0.0

def matched(target):
    lo, hi, min_size = 0.0, 1.0 + 1e-9, int(target // 4)
    for _ in range(40):  # mean size grows with the threshold
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if JevChunks(mid, min_size).mean_size() < target else (lo, mid)
    return JevChunks(hi, min_size)

def evaluate(chunker):
    r = ev.run(chunker, ef, retrieve=RETRIEVE)
    cs = r["corpora_scores"]
    return {m: [x for c in sorted(cs) for x in cs[c][f"{m}_scores"]] for m in ("recall", "precision", "iou")}

t0 = time.time()
results = []
for name, ch in baselines.items():
    jc = matched(np.mean(base_sizes[name]))
    jsizes = [ntok(c) for t in corpora.values() for c in jc.split_text(t)]
    results.append({"name": name, "base": evaluate(ch), "jev": evaluate(jc), "jc": jc,
                    "base_size": np.mean(base_sizes[name]), "jev_size": np.mean(jsizes), "jev_chunks": len(jsizes)})
print(f"{2 * len(results)} benchmark runs in {time.time() - t0:.0f}s")

12 benchmark runs in 447s


## 5. Results

Each row compares one baseline with Jev's chunks at the same mean size. Changes are Jev minus the baseline, with a 95% paired-bootstrap interval over the 472 questions.

In [7]:
label = "Jev" if USE_JEV else "stand-in"
rows = [f"| baseline | mean tokens (baseline / {label}) | recall | IoU | {label} − baseline, IoU (95% CI) | {label} − baseline, recall (95% CI) | threshold | judged cuts |",
        "|---|---|---|---|---|---|---|---|"]
for r in results:
    b, j = r["base"], r["jev"]
    di = paired_bootstrap(b["iou"], j["iou"])
    dr = paired_bootstrap(b["recall"], j["recall"])
    r["d_iou"], r["d_recall"] = di, dr
    rows.append(f"| {r['name']} | {r['base_size']:.0f} / {r['jev_size']:.0f} "
                f"| {np.mean(b['recall']):.3f} → {np.mean(j['recall']):.3f} | {np.mean(b['iou']):.4f} → {np.mean(j['iou']):.4f} "
                f"| {di[0]:+.4f} ({di[1]:+.4f} to {di[2]:+.4f}) | {dr[0]:+.3f} ({dr[1]:+.3f} to {dr[2]:+.3f}) "
                f"| {r['jc'].threshold:.3f} | {r['jc'].judged_share():.0%} |")
display(Markdown("\n".join(rows)))

| baseline | mean tokens (baseline / Jev) | recall | IoU | Jev − baseline, IoU (95% CI) | Jev − baseline, recall (95% CI) | threshold | judged cuts |
|---|---|---|---|---|---|---|---|
| fixed 200 | 200 / 201 | 0.836 → 0.904 | 0.0489 → 0.0515 | +0.0026 (-0.0001 to +0.0053) | +0.068 (+0.038 to +0.100) | 0.350 | 91% |
| fixed 400 | 398 / 337 | 0.814 → 0.891 | 0.0254 → 0.0312 | +0.0057 (+0.0046 to +0.0070) | +0.077 (+0.043 to +0.111) | 1.000 | 0% |
| recursive 200 | 137 / 140 | 0.818 → 0.873 | 0.0645 → 0.0671 | +0.0027 (-0.0012 to +0.0071) | +0.055 (+0.020 to +0.089) | 0.200 | 97% |
| recursive 400 | 276 / 280 | 0.833 → 0.885 | 0.0340 → 0.0353 | +0.0014 (-0.0006 to +0.0032) | +0.052 (+0.019 to +0.084) | 0.710 | 69% |
| chonkie semantic, threshold 0.8 | 89 / 95 | 0.767 → 0.837 | 0.0815 → 0.0900 | +0.0085 (+0.0028 to +0.0152) | +0.069 (+0.032 to +0.108) | 0.130 | 100% |
| chonkie semantic, threshold 0.5 | 137 / 140 | 0.801 → 0.878 | 0.0556 → 0.0674 | +0.0118 (+0.0072 to +0.0169) | +0.078 (+0.042 to +0.118) | 0.200 | 97% |

### Is the recall gain just longer retrieved chunks?

Sizes are matched on the mean, but Jev's chunks vary in length and retrieval may favour the long ones. If it did, the retrieved text would grow and precision (the share of retrieved tokens that belong to the answer) would fall. The retrieved length per question follows from the package's own scores: recall ÷ precision is proportional to retrieved tokens, so the per-question ratio between two chunkers needs no access to the retrieved chunks.

In [8]:
rows = [f"| baseline | precision (baseline → {label}) | {label} − baseline, precision (95% CI) | retrieved tokens, {label} ÷ baseline (median per question) |",
        "|---|---|---|---|"]
for r in results:
    b, j = r["base"], r["jev"]
    dp = paired_bootstrap(b["precision"], j["precision"])
    ratio = [(rj / pj) / (rb / pb) for rb, pb, rj, pj in zip(b["recall"], b["precision"], j["recall"], j["precision"]) if min(rb, pb, rj, pj) > 0]
    rows.append(f"| {r['name']} | {np.mean(b['precision']):.4f} → {np.mean(j['precision']):.4f} "
                f"| {dp[0]:+.4f} ({dp[1]:+.4f} to {dp[2]:+.4f}) | {np.median(ratio):.2f} ({len(ratio)} questions) |")
display(Markdown("\n".join(rows)))

| baseline | precision (baseline → Jev) | Jev − baseline, precision (95% CI) | retrieved tokens, Jev ÷ baseline (median per question) |
|---|---|---|---|
| fixed 200 | 0.0492 → 0.0516 | +0.0024 (-0.0002 to +0.0051) | 1.08 (398 questions) |
| fixed 400 | 0.0255 → 0.0312 | +0.0057 (+0.0046 to +0.0070) | 0.91 (378 questions) |
| recursive 200 | 0.0647 → 0.0674 | +0.0027 (-0.0013 to +0.0070) | 1.01 (376 questions) |
| recursive 400 | 0.0340 → 0.0354 | +0.0014 (-0.0006 to +0.0032) | 1.03 (380 questions) |
| chonkie semantic, threshold 0.8 | 0.0829 → 0.0910 | +0.0081 (+0.0025 to +0.0148) | 1.00 (367 questions) |
| chonkie semantic, threshold 0.5 | 0.0559 → 0.0677 | +0.0118 (+0.0073 to +0.0168) | 0.86 (373 questions) |

### Reading the results

In [9]:
better = [r["name"] for r in results if r["d_iou"][1] > 0]
worse = [r["name"] for r in results if r["d_iou"][2] < 0]
lines = []
if not USE_JEV:
    lines.append("**These numbers come from the stand-in, not Jev.** They show that the pipeline runs end to end and how a chunker "
                 "cutting on adjacent-sentence embedding dissimilarity compares. Set `OPENROUTER_API_KEY` to measure Jev.")
lines.append(f"At matched mean size, the {label} chunks have higher IoU than {len(better)} of {len(results)} baselines "
             f"({', '.join(better) or 'none'}) and lower IoU than {len(worse)} ({', '.join(worse) or 'none'}); "
             f"the remaining intervals include zero.")
display(Markdown("\n\n".join(lines)))

At matched mean size, the Jev chunks have higher IoU than 3 of 6 baselines (fixed 400, chonkie semantic, threshold 0.8, chonkie semantic, threshold 0.5) and lower IoU than 0 (none); the remaining intervals include zero.

## What the Jev run shows

Measured on 1 October 2026 with `typesafe/jev-1.13`: 635 requests, $0.092, 61 s with 8 workers.

- **Recall goes up against every baseline.** At matched mean size, the chunks retrieved for a question cover more of its answer: +0.068 against fixed 200 (0.836 → 0.904), +0.055 against recursive 200, +0.052 against recursive 400, +0.069 and +0.078 against chonkie at thresholds 0.8 and 0.5. Every interval is above zero. Scores shift by a few thousandths between identical runs of the benchmark, well inside these intervals.
- **IoU does not drop.** It rises against both chonkie settings (+0.009 and +0.012, intervals above zero); against fixed 200 and both recursive chunkers the intervals include zero.
- **The gain is not longer retrieved chunks.** Precision never drops (higher against both chonkie settings, within noise elsewhere), and the median retrieved length per question is 0.86 to 1.08 times the baseline's: about the same text, more of it on the answer.
- **Jev beats the embedding version of the same idea.** In the offline run, the stand-in (cut where adjacent sentences' embeddings are least similar, which is how semantic chunkers work) lost 0.03 to 0.05 recall against the same baselines. Jev's gap answers gain 0.06 to 0.07.
- **The fixed-400 row is not a comparison.** Jev's chunks average 337 tokens there, with no cut above the threshold (see below).
- **At small sizes the cutoff is low.** Matching a 90 to 200-token mean needs cutoffs of 0.13 to 0.35, so many cuts land on gaps Jev does not rate as topic changes. At those sizes the request picks the least-bad place to cut rather than a topic boundary.


### What this does not show

- **Matched mean, not matched distribution.** Jev's chunks vary in size and the fixed baselines do not, so with 5 chunks retrieved per question the two can retrieve different numbers of tokens.
- **No matched comparison at 400 fixed tokens.** A cut forced by the 512-token cap lands on the likeliest gap in the chunk rather than at the overflow point, so even with no judged cuts Jev's chunks average about 330 tokens. The fixed-400 row is therefore not size-matched (threshold 1.0, 0% judged cuts); read the 200-token, recursive and chonkie rows.
- **One embedding model.** bge-small truncates at 512 tokens, so every chunker is capped near that, and a stronger embedding model can change the ranking.
- **One benchmark, in English.** Five corpora; the per-corpus scores are in `results[i]["base"]` and `["jev"]` in corpus order.
- **No overlap.** Every chunker here cuts without overlap; overlapping windows help fixed-size chunkers most.